In [1]:
import os
import csv
import shutil
import soundfile as sf
import numpy as np

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset


In [2]:
from transformers import Seq2SeqTrainer
from transformers import WhisperForConditionalGeneration
from transformers import WhisperProcessor
from transformers import Seq2SeqTrainingArguments

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [3]:
from evaluate import load as metrics_loader

In [4]:
import torch

In [5]:
import re

def normalize_cs(text):
    text = text.lower()

    # keep English letters + Twi chars
    text = re.sub(r"[^a-z0-9ɔɛ\s']", "", text)

    # normalize apostrophes (optional)
    text = re.sub(r"'", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [6]:
wer_metric = metrics_loader("wer")

def get_wer(references, predictions, normalize=True, verbose=True):
  rs = references
  ps = predictions
  if normalize:
    ps = [normalize_cs(x) for x in predictions]
    rs = [normalize_cs(x) for x in references]
  if verbose:
    for r, p in zip(rs, ps):
      print(r)
      print(p)
      print()

  return wer_metric.compute(references=rs, predictions=ps)

This function counts the number of trainable parameters in the model.

In [7]:
def count_trainable_parameters(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

## Model Training
This cell uses an interactive dropdown widget to select the language for the ASR model.

In [8]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained("devkyle/Akan-tiny-2000ms-1k")
processor = WhisperProcessor.from_pretrained("devkyle/Akan-tiny-2000ms-1k") # processor from base model

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

2.6.0+cu124
True
NVIDIA RTX 4000 SFF Ada Generation


## Extract features on dataset

In [9]:
LANGUAGE = None
TASK = 'transcribe'

print(f"Set LANGUAGE to: {LANGUAGE}")
print(f"Set TASK to: {TASK}")

Set LANGUAGE to: None
Set TASK to: transcribe


In [10]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is: ', device)

# for more efficient dataset processing
torch.set_num_threads(1)
torch.get_num_threads()
num_proc = os.cpu_count()
print('# processors:', num_proc)


device is:  cuda
# processors: 20


In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "Kennethdot/Ghana_English-Twi_Code-switching_Speech",
)

dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)


In [13]:
sample = dataset["validation"][2]

print(sample["transcript"])

“Doctor no se wo blood pressure no ayɛ high kakra, nti ɛsɛ sɛ wo pɛ time rest-i paa. Ɔhwɛɛ ultrasound results no mu and everything shows that the baby is doing well, nanso ɛsɛ sɛ yɛkɔ so a monitor amniotic fluid no yie. Ɛyaa try sɛ wobɛdi vegetables bebree na gyae salty foods for now. Wo next clinic appointment falls on a Monday, nti mma wo werɛ mfi. Yɛbɛ arrange-i car ama wo na w’aba


In [ ]:
from IPython.display import Audio, display

display(Audio(sample["audio"]["array"], rate=16000))

In [14]:
inputs = processor.feature_extractor(
    sample["audio"]["array"],
    sampling_rate=16000,
    return_tensors="pt"
)

# Get the actual tensor
input_features = inputs.input_features.to(model.device)

with torch.no_grad():
    predicted_ids = model.generate(input_features, max_new_tokens=256, task="transcribe")

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
print(transcription)

Dɔta no so a ɛwɔ blur so no ayɛ hand kakra. Nto asesɛ wɔ pɛtam am ɛrɛstepa. Ɔhwɛ ɔɔtwa so no redzeɛ ɛna ɛwɔ dinhyɛw dze de ebi yɛ adwene wɔ hɔ. Na nso asesɛ yɛ kɔ so a hɔn tsɛ ameɔte fla wɔ ne nua. Ɛyɛ atwa ɛhyɛwɔ baabi. Wɔnye dze abɔage bebree no gyina nso a ɔte futfoɔ no. Hɔne ɛsklaa ne akɔentenfoɔs ɔ ne mbɔde nyimma ɛmpra. Nyimma ɔwɔyɛ mfimfini.


In [ ]:
# Calculate model memory footprint

model_size_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Model size: {model_size_mb:.2f} MB")

Model size: 144.05 MB


In [ ]:
# def prepare_dataset(batch, processor = processor):
#     audio = batch["audio"]

#     inputs = processor.feature_extractor(
#         audio["array"],
#         sampling_rate=audio["sampling_rate"],
#         return_tensors="np",
#     )

#     batch["input_features"] = inputs.input_features[0]

#     batch["labels"] = processor.tokenizer(
#         batch["transcript"]
#     ).input_ids

#     batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]

#     return batch


# # Only preprocess train + validation
# processed_ds = {}

# for split in ["train", "validation"]:
#     processed_ds[split] = dataset[split].map(
#         prepare_dataset,
#         remove_columns=dataset[split].column_names,
#         num_proc=1,
#     )

Map: 100%|██████████| 2159/2159 [01:00<00:00, 35.94 examples/s]


In [ ]:
# from datasets import DatasetDict

# processed_ds = DatasetDict(processed_ds)

# processed_ds.save_to_disk("./processed_dataset")

Saving the dataset (5/5 shards): 100%|██████████| 2159/2159 [00:09<00:00, 236.15 examples/s]


In [15]:
from datasets import load_from_disk
my_audio_dataset = load_from_disk("./processed_dataset")
my_audio_dataset.set_format("torch")

Loading dataset from disk:   0%|          | 0/98 [00:00<?, ?it/s]

In [16]:
print(my_audio_dataset['train'])

Dataset({
    features: ['input_features', 'labels', 'input_length'],
    num_rows: 50965
})


## Configure Training

In [17]:
model.config.use_cache = False

In [25]:
print(next(model.parameters()).device)

cuda:0


In [27]:
#@title Training Hyper Parameters
OUTPUT_DIR = './whisper_tuning_akan' #@param
LOG_DIR = os.path.join(OUTPUT_DIR, 'logs')

LEARNING_RATE = 1e-5 #@param
MAX_EPOCHS = 20 #@param
WARMUP_STEPS = 100 #@param
# set this as short as possible for your data
MAX_GEN_LEN = 32 #@param
# if save steps is 0, only last and best model will be written
SAVE_STEPS = 500 #@param

# see
# https://huggingface.co/docs/transformers/v4.46.2/en/main_classes/trainer#transformers.TrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=OUTPUT_DIR + '/logs',
    per_device_train_batch_size=16,      # increase if VRAM allows
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    fp16=True,
    num_train_epochs=MAX_EPOCHS,
    #
    lr_scheduler_type='constant_with_warmup',
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    #
    eval_strategy="steps",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=MAX_GEN_LEN,
    eval_steps=500,
    metric_for_best_model="wer",
    greater_is_better=False,
    #
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_steps=10,
    # report_to=["tensorboard"],
    load_best_model_at_end=True,
    #
    push_to_hub=False,
    remove_unused_columns=False,
    save_total_limit=3,
    dataloader_num_workers=0,      # parallel data loading
    dataloader_pin_memory=False,
    #eval_on_start=True,
)

Since we are using an `akan-whisper-model`, we need to explicitly define the `LANGUAGE` and `TASK` variables for subsequent operations.

In [20]:
#@title define which parameters to update
#@markdown For personalizartion, we typically only want to update the encoder and projection layer.
#@markdown Updating the decoder layer may lead to overfitting.
UPDATE_ENCODER = True #@param{type: 'boolean'}
UPDATE_DECODER = False #@param{type: 'boolean'}
UPDATE_PROJ = True #@param{type: 'boolean'}
model.model.encoder.requires_grad_(UPDATE_ENCODER)
model.model.decoder.requires_grad_(UPDATE_DECODER)
model.proj_out.requires_grad_(UPDATE_PROJ)


print('encoder params to update/total:', count_trainable_parameters(model.model.encoder), model.model.encoder.num_parameters())
print('decoder parans to update/total:', count_trainable_parameters(model.model.decoder), model.model.decoder.num_parameters())

print('overall # trainable parameters:', count_trainable_parameters(model))
print('overall # model parameters:', model.model.num_parameters())

encoder params to update/total: 8208384 8208384
decoder parans to update/total: 19916160 29552256
overall # trainable parameters: 28124544
overall # model parameters: 37760640


In [28]:
#@title Define Trainer
import evaluate
metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=my_audio_dataset["train"],
    eval_dataset=my_audio_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


## Run Training

In [23]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [29]:
trainer.train()

  0%|          | 0/15920 [00:00<?, ?it/s]

{'loss': 0.8844, 'grad_norm': 5.439831256866455, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}
{'loss': 0.8771, 'grad_norm': 6.035254955291748, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.03}
{'loss': 1.0232, 'grad_norm': 5.649024486541748, 'learning_rate': 3e-06, 'epoch': 0.04}
{'loss': 0.9082, 'grad_norm': 5.293950080871582, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.05}
{'loss': 0.8327, 'grad_norm': 5.283947944641113, 'learning_rate': 5e-06, 'epoch': 0.06}
{'loss': 0.8557, 'grad_norm': 5.098110675811768, 'learning_rate': 6e-06, 'epoch': 0.08}
{'loss': 0.8347, 'grad_norm': 5.043375492095947, 'learning_rate': 7e-06, 'epoch': 0.09}
{'loss': 0.8424, 'grad_norm': 4.834325313568115, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.1}
{'loss': 0.8475, 'grad_norm': 5.384091377258301, 'learning_rate': 9e-06, 'epoch': 0.11}
{'loss': 0.762, 'grad_norm': 5.006831169128418, 'learning_rate': 1e-05, 'epoch': 0.13}
{'loss': 0.7832, 'grad_norm': 5.279120922088623, 'learni

  0%|          | 0/540 [00:00<?, ?it/s]

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}


{'eval_loss': 0.6589933037757874, 'eval_wer': 76.92617400534387, 'eval_runtime': 179.5726, 'eval_samples_per_second': 12.023, 'eval_steps_per_second': 3.007, 'epoch': 0.63}


...

c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.011, 'grad_norm': 0.7555245161056519, 'learning_rate': 1e-05, 'epoch': 19.47}
{'loss': 0.0154, 'grad_norm': 0.7579153776168823, 'learning_rate': 1e-05, 'epoch': 19.49}
{'loss': 0.0115, 'grad_norm': 0.6983094811439514, 'learning_rate': 1e-05, 'epoch': 19.5}
{'loss': 0.0139, 'grad_norm': 1.0140992403030396, 'learning_rate': 1e-05, 'epoch': 19.51}
{'loss': 0.0141, 'grad_norm': 1.1754648685455322, 'learning_rate': 1e-05, 'epoch': 19.52}
{'loss': 0.0141, 'grad_norm': 0.7915241718292236, 'learning_rate': 1e-05, 'epoch': 19.54}
{'loss': 0.0149, 'grad_norm': 1.1699222326278687, 'learning_rate': 1e-05, 'epoch': 19.55}
{'loss': 0.0115, 'grad_norm': 0.6356792449951172, 'learning_rate': 1e-05, 'epoch': 19.56}
{'loss': 0.0136, 'grad_norm': 1.010809063911438, 'learning_rate': 1e-05, 'epoch': 19.57}
{'loss': 0.0151, 'grad_norm': 0.5614888072013855, 'learning_rate': 1e-05, 'epoch': 19.59}
{'loss': 0.0122, 'grad_norm': 0.8239715099334717, 'learning_rate': 1e-05, 'epoch': 19.6}
{'loss': 0.012

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


{'loss': 0.0159, 'grad_norm': 1.2672982215881348, 'learning_rate': 1e-05, 'epoch': 19.99}
{'train_runtime': 21433.5909, 'train_samples_per_second': 47.556, 'train_steps_per_second': 0.743, 'train_loss': 0.12346935311107704, 'epoch': 19.99}


TrainOutput(global_step=15920, training_loss=0.12346935311107704, metrics={'train_runtime': 21433.5909, 'train_samples_per_second': 47.556, 'train_steps_per_second': 0.743, 'total_flos': 2.507854432407552e+19, 'train_loss': 0.12346935311107704, 'epoch': 19.987445072190834})

In [30]:
print('evaluating best model after fine-tuning, lanuage:', LANGUAGE)
# Let Whisper handle multilingual input 
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

results = trainer.evaluate(my_audio_dataset["validation"].shuffle(seed=42).select(range(10)))
print(results)

evaluating best model after fine-tuning, lanuage: None


  0%|          | 0/3 [00:00<?, ?it/s]

{'eval_loss': 0.061905860900878906, 'eval_wer': 69.68641114982579, 'eval_runtime': 1.1492, 'eval_samples_per_second': 8.702, 'eval_steps_per_second': 2.61, 'epoch': 19.987445072190834}


In [31]:
output_dir = 'finetuned_whisper_model_akan' #@param {type: 'string'}
!mkdir -p {output_dir}

print('Saving model in:', output_dir)

# save model and processor, so we can later load as pretrained
save_model_dir = os.path.join(output_dir, 'saved_model')
trainer.model.save_pretrained(save_model_dir, safe_serialization=False)

# save processor also
save_processor_dir = os.path.join(output_dir, 'saved_processor')
processor.save_pretrained(save_processor_dir, safe_serialization=False)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}


Saving model in: finetuned_whisper_model_akan


[]

# Test adapted model

In [32]:
def transcribe_from_dataset(dataset_sample, whisper_model, max_new_tokens=128):
  input_features = processor.feature_extractor(
    dataset_sample["array"],
    sampling_rate=dataset_sample["sampling_rate"],
    return_tensors="pt").input_features

  predicted_ids = whisper_model.generate(
      input_features, max_new_tokens=max_new_tokens,
      task=TASK, forced_decoder_ids=None)
  transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
  return transcription[0].strip()

In [ ]:
default_model = WhisperForConditionalGeneration.from_pretrained("GiftMark/akan-whisper-model")
default_processor = WhisperProcessor.from_pretrained("GiftMark/akan-whisper-model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Kennethdot/Ghana_English-Twi_Code-switching_ASR",
                       )

my_audio_dataset = dataset

In [ ]:
my_audio_dataset = my_audio_dataset.cast_column("audio", Audio(sampling_rate=16000))


In [ ]:
#@title Get WER on default and tuned model for comparison

save_pretrained_model_dir = save_model_dir # Use the already defined save_model_dir

finetuned_model = WhisperForConditionalGeneration.from_pretrained(save_pretrained_model_dir, local_files_only=True)

num_test_samples = 20 #@param{type: 'number'}
normalize_for_wer_calc = True #@param{type: 'boolean'}

num_test_samples = min(num_test_samples, len(my_audio_dataset['test']))
print('number of test examples to process:', num_test_samples)

predictions = []
finetuned_predictions = []
references  = []

for idx in range(num_test_samples):
  print('inference on example:', idx)
  sample = my_audio_dataset['test'][idx]["audio"]
  predictions.append(transcribe_from_dataset(sample, default_model))
  finetuned_predictions.append(transcribe_from_dataset(sample, finetuned_model))
  references.append(my_audio_dataset['test'][idx]['transcript'])

default_wer = get_wer(references=references, predictions=predictions, normalize=normalize_for_wer_calc, verbose=False)
finetuned_wer = get_wer(references=references, predictions=finetuned_predictions, normalize=normalize_for_wer_calc, verbose=False)

print(f'DEFAULT WER: {default_wer}')
print(f'FINETUNED WER: {finetuned_wer}')

number of test examples to process: 20
inference on example: 0
inference on example: 1
inference on example: 2
inference on example: 3
inference on example: 4
inference on example: 5
inference on example: 6
inference on example: 7
inference on example: 8
inference on example: 9
inference on example: 10
inference on example: 11
inference on example: 12
inference on example: 13
inference on example: 14
inference on example: 15
inference on example: 16
inference on example: 17
inference on example: 18
inference on example: 19
DEFAULT WER: 0.8698224852071006
FINETUNED WER: 0.047337278106508875


In [ ]:
#@title Run inference on individual example of test set
for idx in range(1000,1020):
  print('inference on example:', idx)
  sample = my_audio_dataset['test'][idx]["audio"]
  transcript = my_audio_dataset['test'][idx]["transcript"]
  # use default_model or finetuned_model
  model = finetuned_model
  pred = transcribe_from_dataset(sample, model, max_new_tokens=32)
  print('Ground truth: ', transcript)
  print('  Prediction: ', pred)

inference on example: 1000
Ground truth:  Twitwa enam no into small pieces for the light soup.
  Prediction:  Twitwa enam no into small pieces for the light soup.
inference on example: 1001
Ground truth:  I just realized that w'abusua yɛ Ɔyoko, so you are royalty.
  Prediction:  I just realized that w'abusua yɛ Ɔyoko, so you are royalty.
inference on example: 1002


c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\ve

Ground truth:  It is an akyiwadeɛ for an Aduana person to eat certain meats.
  Prediction:  It is an akyiwadeɛ for an Aduana person to eat certain meats.
inference on example: 1003


c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\ve

Ground truth:  “Wo yɛ sure sɛ  wo werɛ mfi?”
  Prediction:  “Wo yɛ sure sɛ ɛw werɛ mfi?”
inference on example: 1004


...

Ground truth:  The fishermen sang 'Sisiri mbo, ɔtabon mbo' while at sea.
  Prediction:  The fishermen sang sesiri mbo ɔtabon mbo while at sea.
inference on example: 1005
Ground truth:  M’asende wo receipt no, have you seen it?
  Prediction:  M’asende wo receipt no, have you seen it?
inference on example: 1006


c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\ve

Ground truth:  Wo nim sɛ I almost forgot to buy the food?
  Prediction:  Wo nim sɛ I almost forgot to buy the food?
inference on example: 1007


c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\transformers\generation\configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\KASA\Downloads\Kasa\ve

Ground truth:  I'm feeling dehydrated, ma me chilled water kakra.
  Prediction:  I'm feeling dehydrated, ma ma chilled water kakra.
inference on example: 1008
Ground truth:  Gyae saa argument no, it's not that deep.
  Prediction:  Gyae saa argument no, it’s not that deep.
inference on example: 1009
Ground truth:  M’asende wo file no, did you see it?
  Prediction:  M’asende wo file no, did you see it?
inference on example: 1010
Ground truth:  I'm trying to upload the video, but network no ayɛ slow.
  Prediction:  I'm trying to upload the video, but network no ayɛ slow.
inference on example: 1011
Ground truth:  Car no wɔ hen? I have been standing here long.
  Prediction:  Car no wɔ hen? I’ve been standing here long.
inference on example: 1012
Ground truth:  Maba ha dadaada, where were you?.
  Prediction:  Maba ha dadaada, where were you?
inference on example: 1013


In [ ]:
import torch

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
save_pretrained_model_dir = "./finetuned_whisper_model_akan/saved_model"
# processor = WhisperProcessor.from_pretrained("GiftMark/akan-whisper-model")
save_processor_dir = "./finetuned_whisper_model_akan/saved_processor"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import gradio as gr
import torch
import torchaudio
import numpy as np
import soundfile as sf

# =========================
# LOAD MODEL (LOCAL)
# =========================
model = WhisperForConditionalGeneration.from_pretrained(
    save_pretrained_model_dir,
    local_files_only=True
)
# processor = WhisperProcessor.from_pretrained(
#     save_processor_dir,
#     local_files_only=True
# )

model = model.to(device)
model.eval()

# =========================
# TRANSCRIPTION FUNCTION
# =========================
def transcribe_audio(audio_path):
    if audio_path is None:
        return "No audio provided."

    try:
        audio_data, sampling_rate = sf.read(audio_path)

        # Fix stereo → mono early (before resampling)
        if len(audio_data.shape) > 1:
            audio_data = np.mean(audio_data, axis=1)

        # Convert to float32 early
        audio_data = audio_data.astype(np.float32)

        if sampling_rate != 16000:
            audio_tensor = torch.tensor(audio_data, dtype=torch.float32)  # ✅ fix here
            resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
            audio_data = resampler(audio_tensor).numpy()
            sampling_rate = 16000

    except Exception as e:
        return f"Error reading audio: {e}"

    # Normalize
    if np.max(np.abs(audio_data)) > 0:
        audio_data = audio_data / np.max(np.abs(audio_data))

    # Extract features
    input_features = processor.feature_extractor(
        audio_data,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).input_features.to(device)

    # Generate transcription
    with torch.no_grad():
        generated_ids = model.generate(
            input_features,
            task="transcribe",
            language="yo",
            temperature=0.0
        )

    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    return transcription

# =========================
# GRADIO UI (SIMPLE)
# =========================
gradio_interface = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(
        sources=["microphone", "upload"],  # BOTH supported
        type="filepath",
        label="Record or Upload Audio"
    ),
    outputs=gr.Textbox(label="Transcription"),
    title="Akan + English Speech Transcription",
    description="Record or upload audio to transcribe Akan and English speech.",
)

# =========================
# LAUNCH
# =========================
if __name__ == "__main__":
    gradio_interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://09598f340b702246ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
